[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Tables and Metadata


## What you will be able to do

Describe a database's tables in Python with `Table`, `Column` and `MetaData`, with primary keys,
foreign keys, and unique and check constraints, and read the `CREATE TABLE` each one becomes. Give
every constraint a predictable name, create the tables and load them, and choose between a default
that SQLAlchemy supplies and one the database does. Read the tables of a database somebody else
designed without writing them out, and recognize the errors a second definition, a duplicate, a
missing default and a misspelled table produce.


## The idea

### The problem

So far the college's tables have existed only as SQL text: a string of `CREATE TABLE` statements in
Setup, handed to `sqlite3` and forgotten. Python knows nothing about them. A query that names a
column that is not there fails only when it runs, a `Date` column cannot turn text into a `date`
because no program has said the column is a date, and every notebook's queries repeat the table and
column names by hand. The **Why SQLAlchemy** notebook built one `Table` for its search, and it was
the only part of the college that Python could check.

The constraints have a problem of their own. The `UNIQUE` on `students.email` and the four foreign
keys were written without names, and a database gives such a constraint a name of its own choosing,
or none at all. When the registrar asks for the rule that stops two students sharing an email to be
changed, a migration has to find that constraint by name, and on SQLite it cannot. And a database
that another office designed, such as the admissions office's, has tables that are there to be read,
and nobody wants to copy fifty column definitions out of it by hand. This notebook treats the
database from Setup as one of those.

### What a table description is

> A **`MetaData`** is a collection of **`Table`** objects, the schema of a database written in
> Python. A `Table` has a name, **`Column`** objects, each with a name, a type such as `Integer` or
> `String(100)`, and settings such as `nullable=False` and `primary_key=True`, and **constraints**:
> a **`ForeignKey`** on a column, **`UniqueConstraint`** and **`CheckConstraint`**. A MetaData's
> **naming convention** gives every constraint a name built from a pattern, such as
> `uq_%(table_name)s_%(column_0_N_name)s`. **`metadata.create_all(engine)`** creates the tables a
> database does not have yet, and **reflection**, with `Table(name, metadata, autoload_with=engine)`
> or **`inspect(engine)`**, reads a table's description out of a database that already has it.

### Why it works that way

- **A `Table` is a description, not a connection.** It can be built with no database at all, and
  printed as the `CREATE TABLE` for any dialect, and a statement built from it can name only the
  columns it has.
- **`create_all` creates what is missing, and nothing else.** It asks the database which tables
  exist, creates the rest in an order where every table comes after the tables it refers to, and
  never changes a table that is already there. Running it twice is safe, and changing a table that
  is in use is a migration's job.
- **A `MetaData` holds one `Table` for every name.** A second definition under the same name is
  refused, which is exactly what running a notebook cell again produces.
- **A constraint with a name can be found again.** SQLite names a failed check by its name, and a
  migration drops or replaces a constraint by name. A naming convention names every constraint the
  same way, so nobody has to remember to.
- **A default lives in Python or in the table.** `default=` is a value SQLAlchemy puts into its own
  `INSERT` statements, and `server_default=` becomes part of the `CREATE TABLE`, so the database
  applies it to every insert, whoever sends it.
- **Reflection reads what the database already knows.** SQLite keeps every table's definition, and
  `inspect` asks for columns, keys and constraints, or builds whole `Table` objects from them.

### Where this shows up

Every ORM class builds a `Table` behind it, so the **Declarative Models** notebook writes the same
schema as classes, and `Base.metadata` is a `MetaData` like this one. Alembic, the subject of the
**Migrations with Alembic** notebook, compares a `MetaData` with a database to write migrations, and
on SQLite it can change a constraint only if the constraint has a name, which is why this notebook
names them all. A report against a database that another team owns reads its tables with
reflection. The **Constraints** notebook of the **sqlite3, Deep Dive** guide wrote the same kinds of
constraint in SQL.

### What this notebook covers

- A `Table`, its columns and the `CREATE TABLE` it becomes
- The college's five tables, with constraints that have names
- Creating the tables, and creating them again
- Loading them, and what a named constraint says when it fails
- A default in Python, or a default in the table
- Reflecting a database somebody else designed
- Which way to describe a table, and which default to give it
- The college, built from its `MetaData` and checked, finished
- Five errors, from a table defined twice to a table that is not there

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import Column, Integer, MetaData, String, Table, create_engine, inspect

metadata = MetaData(naming_convention={"uq": "uq_%(table_name)s_%(column_0_name)s"})
courses = Table("courses", metadata,
                Column("id", Integer, primary_key=True),
                Column("code", String(10), nullable=False, unique=True),
                Column("credits", Integer, nullable=False))

engine = create_engine("sqlite://")
metadata.create_all(engine)
print(inspect(engine).get_table_names())
print(inspect(engine).get_unique_constraints("courses"))
```

```
['courses']
[{'name': 'uq_courses_code', 'column_names': ['code']}]
```

The table was described in Python, and `create_all` wrote it to the database. `inspect` then read
the database back, and found the unique constraint under the name the naming convention gave it.


## Setup

Ten imports, the college's database, and the engine helper.

- `sqlalchemy` is the library itself, and the cell prints its version
- `MetaData`, `Table`, `Column`, the types `Integer`, `String` and `Date`, `ForeignKey`,
  `UniqueConstraint` and `CheckConstraint`, from `sqlalchemy`, describe tables
- `inspect` reads a database's tables back, `insert`, `select` and `func` load and count rows, and
  `create_engine`, `event` and `text` make the engine and run SQL
- `CreateTable`, from `sqlalchemy.schema`, turns a `Table` into its `CREATE TABLE` statement
- `IntegrityError` and `OperationalError`, from `sqlalchemy.exc`, are the errors raised by a broken
  constraint and by a column the database does not have
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what a `Date` column takes and returns
- `logging` carries the SQL an engine logs to `PrintStatements`
- `sqlite3` builds the database, `Path` names the files, and `shutil` removes the scratch folder at
  the start and at the end

The database, `scratch/college.db`, is the one the **Connections and Transactions** notebook built,
from SQL text with `sqlite3`, and this notebook treats it as a database somebody else designed.
`PrintStatements` and `college_engine` are the handler and the engine helper from the
**Engines and URLs** notebook, the helper as the **Connections and Transactions** notebook left it.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
import sqlite3
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        create_engine, event, func, insert, inspect, select, text)
from sqlalchemy.exc import IntegrityError, OperationalError
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", len(STUDENTS), "students,", len(ENROLLMENTS), "enrollments")


sqlalchemy 2.0.54 | scratch/college.db | 25 students, 228 enrollments


## Worked examples

### A Table, and the CREATE TABLE it becomes

A `Table` takes its name, the `MetaData` it belongs to, and its columns. `show_ddl` prints the
`CREATE TABLE` statement a table becomes for an engine's database, which is data definition
language, the part of SQL that describes tables:


In [2]:
def show_ddl(table, engine):
    """Print the CREATE TABLE statement a Table becomes on an engine's database."""
    for line in str(CreateTable(table).compile(engine)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))



sketch = MetaData()
students = Table(
    "students", sketch,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)

memory = college_engine()
print("columns:    ", students.c.keys())
print("primary key:", students.primary_key.columns.keys(), "| email unique:", students.c.email.unique)
print("started_on: ", repr(students.c.started_on.type))
try:
    select(students.c.phone)
except AttributeError as error:
    print("no column", error, "| the query failed before any SQL was written")
show_ddl(students, memory)


columns:     ['id', 'name', 'email', 'program', 'started_on']
primary key: ['id'] | email unique: True
started_on:  Date()
no column phone | the query failed before any SQL was written
    CREATE TABLE students (
        id INTEGER NOT NULL,
        name VARCHAR(100) NOT NULL,
        email VARCHAR(200) NOT NULL,
        program VARCHAR(50) NOT NULL,
        started_on DATE NOT NULL,
        PRIMARY KEY (id),
        UNIQUE (email)
    )


`students.c` holds the columns, by name, and a statement built from the table reaches them there, as
the **Why SQLAlchemy** notebook's search did. A column the table does not have is an
`AttributeError` in Python, raised while the query is being built, rather than an error from the
database after it was sent. `String(100)` became `VARCHAR(100)`: SQLite ignores the
length, and PostgreSQL and MySQL enforce it. `Date` became `DATE`, which SQLite stores as text and
SQLAlchemy turns into a `date` on the way out. `nullable=False` is `NOT NULL`, and the primary key
and the unique email came out as constraints of their own at the end, with no names. `memory` is a
database in memory from `college_engine`, which the tables below are created in.

### The college's five tables, with constraints that have names

The whole schema, in a `MetaData` with a naming convention. Every key in `NAMING` is a kind of
constraint, and its pattern builds the constraint's name from the table and its columns.
`ForeignKey("courses.id")` makes a column refer to another table's, and takes its type from the
column it refers to:


In [3]:
NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

print(len(college.tables), "tables:", list(college.tables))


5 tables: ['students', 'courses', 'terms', 'sections', 'enrollments']


`pk` names the primary key, `uq` a unique constraint, `ck` a check, `fk` a foreign key and `ix` an
index. `%(column_0_N_name)s` joins the names of every column in the constraint, and
`%(constraint_name)s` is the name the constraint was given, which is why a check needs one:
`credits_range` becomes `ck_courses_credits_range`. The naming convention is the pattern the
SQLAlchemy documentation recommends for a schema that Alembic will manage. The cell defines
`students` again, now in `college`, where every name is predictable:


In [4]:
show_ddl(sections, memory)
show_ddl(enrollments, memory)


    CREATE TABLE sections (
        id INTEGER NOT NULL,
        course_id INTEGER NOT NULL,
        term_id INTEGER NOT NULL,
        capacity INTEGER NOT NULL,
        CONSTRAINT pk_sections PRIMARY KEY (id),
        CONSTRAINT uq_sections_course_id_term_id UNIQUE (course_id, term_id),
        CONSTRAINT ck_sections_capacity_positive CHECK (capacity > 0),
        CONSTRAINT fk_sections_course_id_courses FOREIGN KEY(course_id) REFERENCES courses (id),
        CONSTRAINT fk_sections_term_id_terms FOREIGN KEY(term_id) REFERENCES terms (id)
    )
    CREATE TABLE enrollments (
        student_id INTEGER NOT NULL,
        section_id INTEGER NOT NULL,
        status VARCHAR(20) DEFAULT 'enrolled' NOT NULL,
        grade VARCHAR(2),
        CONSTRAINT pk_enrollments PRIMARY KEY (student_id, section_id),
        CONSTRAINT ck_enrollments_status_known CHECK (status IN ('enrolled', 'completed', 'withdrawn')),
        CONSTRAINT fk_enrollments_student_id_students FOREIGN KEY(student_id) REFEREN

Every constraint came out with a name: the primary keys, the unique pair of course and term, the
checks and both foreign keys of every table. `enrollments` has a primary key of two columns, so a
student can be in a section only once, and `status` has `DEFAULT 'enrolled'` written into the table,
which a section below comes back to.

### Creating the tables, and creating them again

`create_all` creates every table of a `MetaData` that the database does not have yet, in an order
where every table comes after the tables it refers to. `college.sorted_tables` is that order. The
second call runs with `echo`, to show what it sends:


In [5]:
college.create_all(memory)
print("tables:           ", inspect(memory).get_table_names())
print("in creation order:", [table.name for table in college.sorted_tables])

memory.echo = True
college.create_all(memory)                    # a second time, with every table already there
memory.echo = False


tables:            ['courses', 'enrollments', 'sections', 'students', 'terms']
in creation order: ['courses', 'students', 'terms', 'sections', 'enrollments']
    BEGIN (implicit)
    PRAGMA main.table_info("students")
    PRAGMA main.table_info("courses")
    PRAGMA main.table_info("terms")
    PRAGMA main.table_info("sections")
    PRAGMA main.table_info("enrollments")
    COMMIT


The second call asked about every table with `PRAGMA main.table_info`, found all five, and created
nothing, so `create_all` is safe to run twice. `inspect(engine)` returns an inspector, whose
`get_table_names()` reads the tables straight from the database. The order puts `courses`,
`students` and `terms` before `sections`, and `sections` before `enrollments`, because a foreign key
can only refer to a table that exists.

### Loading the tables, and a check that fails by name

`insert(courses)` is the statement for adding rows to a `Table`, the subject of the
**SQL Expressions** notebook, and a list of dictionaries runs it once for every dictionary. A course
worth 12 credits breaks `credits_range`:


In [6]:
with memory.begin() as conn:
    conn.execute(insert(courses), [{"code": code, "title": title, "department": department, "credits": credits}
                                   for code, title, department, credits in COURSES])

try:
    with memory.begin() as conn:
        conn.execute(insert(courses).values(code="ART-100", title="Drawing", department="Art", credits=12))
except IntegrityError as error:
    print("refused:", error.orig)

print(inspect(memory).get_check_constraints("courses"))


refused: CHECK constraint failed: ck_courses_credits_range
[{'sqltext': 'credits BETWEEN 1 AND 6', 'name': 'ck_courses_credits_range'}]


Ten courses went in, and the eleventh was refused with the name of the check it broke,
`ck_courses_credits_range`, which says what went wrong without anybody looking up the schema. The
inspector reads the check back from the database with its name and its SQL. A unique constraint is
reported by its columns instead, as Common errors shows.

### A default in Python, or a default in the table

`enrollments.status` has `server_default="enrolled"`, so its `CREATE TABLE` says
`DEFAULT 'enrolled'`, and the database fills the status in for any insert that leaves it out, even
SQL written by hand. The student, term and section go in first, because the enrollment's foreign
keys refer to them:


In [7]:
with memory.begin() as conn:
    conn.execute(insert(students).values(name="Ana Reyes", email="areyes@college.edu", program="Biology",
                                         started_on=date(2024, 8, 26)))
    conn.execute(insert(terms).values(name="Spring 2026", starts_on=date(2026, 1, 12)))
    conn.execute(insert(sections).values(course_id=1, term_id=1, capacity=30))
    conn.execute(text("INSERT INTO enrollments (student_id, section_id) VALUES (1, 1)"))     # no status given
    print(conn.execute(text("SELECT student_id, section_id, status, grade FROM enrollments")).all())


[(1, 1, 'enrolled', None)]


The `INSERT` named no status, and the row has one, supplied by the database itself. `grade` has no
default and allows `NULL`, so it came back as `None`. The other kind, `default="enrolled"`, is a
value SQLAlchemy adds to its own `INSERT` statements, which suits a value only Python can compute,
such as the time of day, and leaves the table without a default of its own. Common errors shows what
that costs when another program inserts. A date goes into a `Date` column as a `date`, and
`date(2024, 8, 26)` is Ana Reyes's first day.

### Reflecting a database you did not design

The Setup cell's database was made from SQL text, and nothing in this notebook described its tables.
Reflection reads them from the database: `MetaData.reflect` builds a `Table` for every table there,
and `autoload_with` builds one:


In [8]:
disk = college_engine(DATABASE)
seen = MetaData()
seen.reflect(disk)
print("tables:", sorted(seen.tables))

found_students = Table("students", MetaData(), autoload_with=disk)
for column in found_students.columns:
    print(f"   {column.name:<11} {column.type!r:<10} nullable={column.nullable}")

with disk.connect() as conn:
    query = select(found_students.c.name, found_students.c.started_on).where(found_students.c.program == "History")
    print(conn.execute(query.order_by(found_students.c.name).limit(2)).all())


tables: ['courses', 'enrollments', 'sections', 'students', 'terms']
   id          INTEGER()  nullable=True
   name        TEXT()     nullable=False
   email       TEXT()     nullable=False
   program     TEXT()     nullable=False
   started_on  TEXT()     nullable=False
[("Aoife O'Brien", '2024-08-26'), ('Elena Petrova', '2025-01-13')]


Five tables, and the columns of `students` as SQLite recorded them: `INTEGER` and `TEXT`, the types
the SQL text gave them. The reflected table builds queries like any other, but `started_on` came
back as text, because reflection can only report the type the database recorded, and nobody told
this database that the column holds dates. A reflected `Table` knows the names, and a `Table`
written in Python knows the meaning. `id` shows `nullable=True`, since the SQL text never said
`NOT NULL`, though SQLite fills in a primary key of type `INTEGER` for itself. The inspector reads
the constraints too:


In [9]:
report = inspect(disk)
print("unique:      ", report.get_unique_constraints("students"))
print("primary key: ", report.get_pk_constraint("enrollments"))
for key in sorted(report.get_foreign_keys("enrollments"), key=lambda key: key["constrained_columns"]):
    print("foreign key: ", key["name"], key["constrained_columns"], "->", key["referred_table"], key["referred_columns"])


unique:       [{'name': None, 'column_names': ['email']}]
primary key:  {'constrained_columns': ['student_id', 'section_id'], 'name': None}
foreign key:  None ['section_id'] -> sections ['id']
foreign key:  None ['student_id'] -> students ['id']


Every constraint is there, and every name is `None`: SQL text without names gave SQLite nothing to
record. A migration on this database cannot drop or change these constraints by name, which is what
the naming convention prevents.

### Which way to describe a table, and which default to give it

| Use | When | Why |
|---|---|---|
| a `Table` written in Python, in a `MetaData` with a naming convention | the schema belongs to your program | the types carry meaning both ways, statements can name only real columns, and every constraint has a name |
| reflection, with `autoload_with` or `MetaData.reflect` | a database somebody else designed, or one you only read | the names come from the database, with no copying, though the types are only what the database recorded |
| `text()` with SQL written by hand | a statement easier to write than to build | nothing to describe, at the price of names no program checks |
| `server_default=` | a value every insert should get, whoever sends it | it is part of the table |
| `default=` | a value only Python can compute, such as `uuid.uuid4` or the time now | SQLAlchemy computes it at every insert of its own |

The default for a schema your program owns is `Table` in Python with a naming convention, and
`server_default=` for any default that is a plain value.

### The college, built from its MetaData, finished

The pieces of this notebook in one function. `build_college` creates the tables of `college` on an
engine, loads the lists from Setup into them in dependency order, turning the dates from text into
`date` objects on the way, and counts the rows. The registrar's database, `scratch/registrar.db`,
gets the college built from its `MetaData`, and the inspector compares its constraint names with
those of the database from Setup:


In [10]:
def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}



registrar = college_engine(SCRATCH / "registrar.db")
print(build_college(registrar))

for label, engine in [("from SQL text:   ", disk), ("from `college`:  ", registrar)]:
    report = inspect(engine)
    names = [str(report.get_pk_constraint("enrollments")["name"])]
    names += sorted(str(key["name"]) for key in report.get_foreign_keys("enrollments"))
    print(label, names)


{'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
from SQL text:    ['None', 'None', 'None']
from `college`:   ['pk_enrollments', 'fk_enrollments_section_id_sections', 'fk_enrollments_student_id_students']


Every row from Setup's lists went in, 228 enrollments with them, and every foreign key held, since
`college_engine` enforces them and `sorted_tables` loaded every table after the ones it refers to.
The same data, in the same shape, and only the database built from `college` can name its
constraints. `select(func.count()).select_from(table)` is `SELECT COUNT(*)` built as a statement,
which the **SQL Expressions** notebook covers.

### Where each part came from

| In `build_college` | What it relies on | The section that showed it |
|---|---|---|
| `college`, with `NAMING` | constraints with predictable names | The college's five tables, with constraints that have names |
| `college.create_all(engine)` | only the missing tables created, in dependency order | Creating the tables, and creating them again |
| `for table in college.sorted_tables` | every table loaded after the tables it refers to | Creating the tables, and creating them again |
| `conn.execute(insert(table), rows[table])` | a list of dictionaries inserted in one call | Loading the tables, and a check that fails by name |
| `date.fromisoformat(...)` | a `Date` column that takes `date` objects | A default in Python, or a default in the table |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/05-tables-and-metadata-solutions.ipynb).

**1.** In a new `MetaData` with the naming convention `NAMING`, describe a `rooms` table: an id, a
`building` and a `number`, both required, and a number of `seats`, with the pair of building and
number unique. Print its `CREATE TABLE`.


In [11]:
# your code here


**2.** Add a check named `seats_positive` to `rooms`, and print the names of all its constraints from
the `Table` itself, with `rooms.constraints`.


In [12]:
# your code here


**3.** Reflect only the `courses` table of the Setup database with `autoload_with`, and print the
name and type of every column.


In [13]:
# your code here


**4.** With `inspect`, list the foreign keys of `sections` in the Setup database: the columns, and
the table and column each one refers to.


In [14]:
# your code here


**5.** Create `rooms` in a database in memory, add two rooms, and try a third that repeats a building
and number. Print the first line of the error.


In [15]:
# your code here


**6.** Drop every table of `college` from the registrar's database with `drop_all`, show that none is
left, and build the college again with `build_college`.


In [16]:
# your code here


## Common errors

### sqlalchemy.exc.InvalidRequestError: Table 'students' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.


In [17]:
roster = MetaData()
Table("students", roster, Column("id", Integer, primary_key=True), Column("name", String(100)))

Table("students", roster, Column("id", Integer, primary_key=True), Column("name", String(100)),
      Column("email", String(200)))


InvalidRequestError: Table 'students' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

A `MetaData` holds one `Table` for every name, and the second definition asked for a table it already
had. In a notebook this comes from running a cell again: a cell that defines a table in a `MetaData`
made by an earlier cell defines it twice. Make the `MetaData` in the same cell as its tables, as this
notebook's cells do, so that running the cell again starts from an empty one. To change a definition
on purpose, `extend_existing=True` replaces its columns:


In [18]:
again = Table("students", roster, Column("id", Integer, primary_key=True), Column("name", String(100)),
              Column("email", String(200)), extend_existing=True)
print(again.c.keys(), "| the same Table object:", again is roster.tables["students"])


['id', 'name', 'email'] | the same Table object: True


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: students.email


In [19]:
with registrar.begin() as conn:
    conn.execute(insert(students).values(name="Ana Reyes", email="areyes@college.edu", program="History",
                                         started_on=date(2026, 1, 12)))


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: students.email
[SQL: INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)]
[parameters: ('Ana Reyes', 'areyes@college.edu', 'History', '2026-01-12')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The unique email did its job: `areyes@college.edu` belongs to a student already, and the second Ana
Reyes was refused. SQLite reports a unique constraint by its columns, `students.email`, not by the
name `uq_students_email`, which is why the column names are what the error gives you to go on. The
program should look first, and decide whether this is the same student:


In [20]:
with registrar.connect() as conn:
    same = select(students.c.id, students.c.name, students.c.program).where(students.c.email == "areyes@college.edu")
    print("already registered:", conn.execute(same).one_or_none())


already registered: (1, 'Ana Reyes', 'Biology')


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: requests.status


In [21]:
drafts = MetaData()
requests = Table(
    "requests", drafts,
    Column("id", Integer, primary_key=True),
    Column("status", String(20), nullable=False, default="pending"),      # a default in Python
)
drafts.create_all(memory)

with memory.begin() as conn:
    conn.execute(insert(requests))                                        # SQLAlchemy fills in the status
    conn.execute(text("INSERT INTO requests (id) VALUES (2)"))            # and here nobody does


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: requests.status
[SQL: INSERT INTO requests (id) VALUES (2)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The first insert ran, because SQLAlchemy put `'pending'` into its own statement. The second was SQL
written by hand, which SQLAlchemy passed on untouched, and it failed, taking the first with it when
the block rolled back. The table itself has no default: its
`CREATE TABLE` says `status VARCHAR(20) NOT NULL`, and nothing more. The same happens to any other
program that writes to the table, such as a nightly import with `sqlite3`. A plain value belongs in
the table, with `server_default`:


In [22]:
drafts.drop_all(memory)
fixed = MetaData()
requests = Table(
    "requests", fixed,
    Column("id", Integer, primary_key=True),
    Column("status", String(20), nullable=False, server_default="pending"),
)
fixed.create_all(memory)

with memory.begin() as conn:
    conn.execute(insert(requests))
    conn.execute(text("INSERT INTO requests (id) VALUES (2)"))
    print(conn.execute(text("SELECT id, status FROM requests")).all())


[(1, 'pending'), (2, 'pending')]


### No error, and no new column: create_all on a table that already exists


In [23]:
growing = MetaData(naming_convention=NAMING)
new_sections = Table(
    "sections", growing,
    Column("id", Integer, primary_key=True),
    Column("course_id", Integer, nullable=False),
    Column("term_id", Integer, nullable=False),
    Column("capacity", Integer, nullable=False),
    Column("room", String(20)),                                           # the new column
)
growing.create_all(registrar)
print("columns in the database:", [column["name"] for column in inspect(registrar).get_columns("sections")])

try:
    with registrar.connect() as conn:
        conn.execute(select(new_sections.c.room))
except OperationalError as error:
    print(str(error).splitlines()[0])


columns in the database: ['id', 'course_id', 'term_id', 'capacity']
(sqlite3.OperationalError) no such column: sections.room


The `Table` gained a `room`, `create_all` ran without an error, and the database has no such column:
`create_all` creates missing tables and never changes one that exists. The mismatch waits until a
query names the column. Adding a column to a table in use is a migration, which is what the
**Migrations with Alembic** notebook is about, and underneath it is an `ALTER TABLE`:


In [24]:
with registrar.begin() as conn:
    conn.execute(text("ALTER TABLE sections ADD COLUMN room VARCHAR(20)"))
print("columns in the database:", [column["name"] for column in inspect(registrar).get_columns("sections")])


columns in the database: ['id', 'course_id', 'term_id', 'capacity', 'room']


### sqlalchemy.exc.NoSuchTableError: student


In [25]:
Table("student", MetaData(), autoload_with=disk)


NoSuchTableError: student

Reflection asked the database for a table called `student`, and there is none: the table is
`students`. The message is only the name, so it is worth reading closely. Ask the database what it
has:


In [26]:
print(inspect(disk).get_table_names())


['courses', 'enrollments', 'sections', 'students', 'terms']


Last, the engines let go of their files and their database in memory, and this cell removes the
scratch folder, with both databases in it:


In [27]:
for engine in (memory, disk, registrar):
    engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A `Table` in a `MetaData` describes a table in Python: columns with types and settings, a primary
  key, foreign keys, and unique and check constraints, and `CreateTable` shows the SQL it becomes.
- A naming convention gives every constraint a predictable name, which a failed check reports and a
  migration needs.
- `create_all` creates the missing tables in dependency order, is safe to run twice, and never
  changes a table that already exists.
- `server_default` puts a default in the table for every insert, and `default` is a value SQLAlchemy
  adds to its own inserts.
- Reflection, with `autoload_with`, `MetaData.reflect` and `inspect`, reads the tables of a database
  somebody else designed, with the names and types it recorded.


## What is next

The **SQL Expressions** notebook builds statements from these tables: `select`, `insert`, `update`
and `delete` as objects, and the `where` clause that quietly lost half of itself.


---

&#8592; **Previous:** [Reading Results](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/04-reading-results.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
